Steps to preprocess MODIS data for phenology tracking. Note: GEE runs into memory issues when you try manipulating any of the big data for plotting/viewing/etc. So I am exporting these as geotiffs and reading them in as rasters to python to be able to look at the processing steps.

Author: Tayler LaSharr

Date: September 2024

In [92]:
## first we will read in necessary python libraries and intialize the GEE module
## you will have to authenticate GEE on your machine to use this code

import ee
import matplotlib.pyplot as plt
from datetime import datetime, timezone
import geopandas as gpd
import pandas as pd
import geemap
import os
import pprint

import json

## if you need to authenticate GEE use this code (will give you a link to allow you to authenticate it online )
# ee.Authenticate()

## initialize the Earth Engine module
ee.Initialize(project='ee-tlasharr')

In [93]:
## set thresholds and things to manipulate

## define the winter period for your study area
start_month_day = '12-01'
end_month_day = '03-30'

## threshold for snow in winter to get the 'baseline' value in winter
snow_threshold = 75

## get the time period we care about
start_date_data = '2015-01-01'
end_date_data = '2023-12-31'


Read in the different data pieces that we care about (modis NDVI, MODIS snow, and mule deer polygons)

In [94]:
## load in a shapefile or area of interest
## this is an assest that is already loaded into GEE

# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/AZ_RainbowValley') ## dry af
md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/WY_Sublette') ## "normal" temperate environment
# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/UT_Paunsaugunt') ## some snow in winter

## get a bounding box for clipping purposes
bounding_box = md_cooridor.geometry().bounds()
small_square_geom = bounding_box  # Use bounding box directly as the clipping region

## load in the MODIS NDVI data and snow data
modis_ndvi = ee.ImageCollection('MODIS/061/MOD13Q1') \
                .filterDate(start_date_data, end_date_data) \
                .select('NDVI', 'DetailedQA') \
                .map(lambda image: image.clip(small_square_geom))

modis_snow = ee.ImageCollection('MODIS/061/MOD10A1') \
                .filterDate(start_date_data, end_date_data) \
                .select('NDSI_Snow_Cover') \
                .map(lambda image: image.clip(small_square_geom))

## first see the projection of the shapefile
coordidor_projection = md_cooridor.first().geometry().projection()
print('Projection of the cooridor shapefile:', coordidor_projection.getInfo())

## now we can reproject the shapefile
modis_projection = modis_ndvi.first().projection() ## this is the projection of the first image of the modis ndvi data
target_crs = modis_projection.crs() ## this is the crs from the modis.projection object
target_crs_scale = modis_projection.nominalScale() ## this is the scale (confused on what this is need to figure this out)

## this is a function to reproject it
def reproject_feature(feature):
    return feature.setGeometry(feature.geometry().transform(target_crs, target_crs_scale))

## this is running the function, so now "reprojected_roi" is the new object we care about
reprojected_roi = md_cooridor.map(reproject_feature)

## make sure the time property is there
def check_time_start(image):
    return ee.Algorithms.If(
        image.get('system:time_start'),
        image,
        image.set('system:time_start', ee.Date(image.date()).millis())
    )

modis_ndvi = modis_ndvi.map(lambda img: ee.Image(check_time_start(img)))



Projection of the cooridor shapefile: {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}


Step 1. Remove low-quality observations with the QA band; this still might need some work. 

In [95]:
## define a function to only return high quality pixels

def quality_mask(image, snow_filter=True):
    ## select the VI_Quality band (not VI_Quality Bitmask if it's not present)
    quality = image.select('DetailedQA')

    ## MODLAND_QA (Bits 0-1) - keep only the pixels where value is 0 or 1 (good quality while also checking other QA things)
    modland_qa = quality.bitwiseAnd(3).lte(1)

    ## VI Usefulness (Bits 2-5) - keep pixels with values <= 2 (higher quality, highest values are worst and it goes to 15, I'm playing with this threshold)
    vi_usefulness = quality.rightShift(2).bitwiseAnd(15).lte(2)

    ## possible Snow/Ice (Bit 14) - mask pixels with snow or ice (might need to get rid of this to deal with winter later??)
    snow_ice = quality.rightShift(14).bitwiseAnd(1).eq(0)

    ## combine all the masks (can turn the snow mask on/off)
    if snow_filter:
        mask = modland_qa.And(vi_usefulness).And(snow_ice)
    else:
        mask = modland_qa.And(vi_usefulness)

    ## apply the mask to the image
    return image.updateMask(mask).select('NDVI')  ## select only NDVI after masking

## apply the mask to the modis NDVI collection
modis_ndvi_filtered = modis_ndvi.map(lambda img: quality_mask(img, snow_filter=False))

Step 2. Set winter baselines dynamically since winter conditions are variable year to year. This differs from the Bischof methods because winter is not consistent and some winters have snow while others do not. For this step we basically go through 3 different substeps. First, we define the winter period, then we calculate the 'baseline' winter values during that period for all cells that have less than a set amount of snow cover. Finally, if there is snow covering the cells in winter, we will replace them with the 'winter baseline'. Finally, to further smooth things in winter, if the image before and after the current image is set to the winter baseline, we will also set the current image to the baselin. This will prevent weird bumps from popping up throuhgout winter.

We won't set the winter values to the "baseline" for the entire winter because in desert systems there might be green that is more variable.

In [96]:
## function to add a property to the image for if it's winter

def filter_to_winter_period(image_collection, start_month_day, end_month_day):
    """Filters the image collection to the given winter time window."""
    
    ## function to check if the image falls within the specified winter period
    def filter_winter(image):
        image_date = ee.Date(image.get('system:time_start'))
        year = image_date.get('year')

        ## create date ranges for the winter period
        start_date = ee.Date.fromYMD(year, ee.Number.parse(start_month_day.split('-')[0]), ee.Number.parse(start_month_day.split('-')[1]))
        end_date = ee.Date.fromYMD(year.add(1), ee.Number.parse(end_month_day.split('-')[0]), ee.Number.parse(end_month_day.split('-')[1]))

        ## apply the date filter to determine if the image falls within the winter period
        is_winter = ee.Algorithms.If(
            image_date.millis().gte(start_date.millis()).And(image_date.millis().lte(end_date.millis())), 
            True, False
        )

        return image.set('is_winter', is_winter)

    ## apply the filter to the image collection and return only winter images
    winter_filtered = image_collection.map(filter_winter).filter(ee.Filter.eq('is_winter', True))

    return winter_filtered

## run the function
modis_ndvi_winter = filter_to_winter_period(modis_ndvi, start_month_day, end_month_day)


In [97]:
## now a function to mask the pixels that have snow above the threshold we care about so we can calculate the 'winter baseline'

def mask_snow_using_modis(image, snow_threshold=50):
    ## get the MODIS NDVI image date
    modis_date = ee.Date(image.get('system:time_start'))

    ## get the closest MODIS snow image after the MODIS NDVI image date
    modis_snow_image = modis_snow.filterDate(modis_date, end_date_data).sort('system:time_start').first()

    ## Check if modis_snow_image exists and contains valid bands
    def apply_snow_mask(image, snow_image):
        snow_mask = snow_image.select('NDSI_Snow_Cover').gt(snow_threshold)
        return image.updateMask(snow_mask.Not())

    ## Check if modis_snow_image exists using ee.Algorithms.If
    return ee.Algorithms.If(
        modis_snow_image,  # Check if there is a valid snow image
        apply_snow_mask(image, modis_snow_image),  # Apply snow mask if snow image exists
        image  # Return original NDVI image if no snow image is found
    )

## apply the snow mask to the winter period NDVI collection
modis_ndvi_winter_no_snow = modis_ndvi_winter.map(lambda img: mask_snow_using_modis(img, snow_threshold))

## then calculate the 0.025 percentile of NDVI values in the snow-masked winter NDVI collection
def calculate_baseline(image_collection, quantile=0.025):
    
    ## reduce the image collection to get the 0.025 percentile
    baseline_ndvi = image_collection.select('NDVI').reduce(ee.Reducer.percentile([quantile * 100])).rename('NDVI_baseline')
    return baseline_ndvi

## calculate the baseline for the winter period
winter_baseline_ndvi = calculate_baseline(modis_ndvi_winter_no_snow, quantile=0.025)

In [101]:
def apply_snow_correction(ndvi_image, modis_snow, snow_threshold, baseline_image):
    """
    Get the closest MODIS snow image and apply the snow mask to adjust the NDVI values based on snow cover.
    """
    ## get the closest MODIS snow image for the NDVI image date
    image_date = ee.Date(ndvi_image.get('system:time_start'))  # Get the NDVI image date

    modis_snow_image_before = modis_snow.filterDate('2000-01-01', image_date).sort('system:time_start', False).first()
    modis_snow_image_after = modis_snow.filterDate(image_date, '2023-12-31').sort('system:time_start').first()

    ## check if both images exist
    def get_valid_image(image_before, image_after):

        ## return the closest snow image based on the difference in days
        return ee.Algorithms.If(
            ee.Algorithms.IsEqual(image_before, None),
            image_after,
            ee.Algorithms.If(
                ee.Algorithms.IsEqual(image_after, None),
                image_before,
                ee.Algorithms.If(
                    image_date.difference(ee.Date(image_before.get('system:time_start')), 'day').abs().lte(
                        image_date.difference(ee.Date(image_after.get('system:time_start')), 'day').abs()),
                    image_before,
                    image_after
                )
            )
        )

    closest_snow_image = ee.Image(get_valid_image(modis_snow_image_before, modis_snow_image_after))

    ## aply the snow mask and adjust NDVI values based on snow cover
    snow_mask = closest_snow_image.select('NDSI_Snow_Cover').gt(snow_threshold)
    adjusted_ndvi = ndvi_image.select('NDVI').where(snow_mask, baseline_image)

    ## return the NDVI image with the adjusted NDVI values
    return ndvi_image.addBands(adjusted_ndvi.rename('NDVI'), overwrite=True)

def apply_baseline_correction_to_collection(image_collection, modis_snow, snow_threshold, baseline_image):
    """
    Apply the baseline correction to the entire NDVI ImageCollection by adjusting
    NDVI values based on snow cover.
    """

    def adjust_ndvi(image):
        ## call the combined function to get the corrected NDVI for each image
        return apply_snow_correction(image, modis_snow, snow_threshold, baseline_image)

    ## map the function over the entire collection
    corrected_collection = image_collection.map(adjust_ndvi)

    return corrected_collection

modis_ndvi_corrected = apply_baseline_correction_to_collection(modis_ndvi_filtered,
                                                               modis_snow,
                                                               snow_threshold=snow_threshold,
                                                               baseline_image=winter_baseline_ndvi)

## then, we're going to smooth things out a bit, if the images t-1 and t+1 are set to the winter baseline, we will also set t to that baseline.

def apply_neighbor_baseline_correction(ndvi_image, baseline_image, all_images):
    """
    Check if both the previous and next images in the collection have been set to the baseline,
    and if so, set the current image to the baseline.
    """
    ## get the current image's index in the sorted collection
    all_images_list = all_images.toList(all_images.size())
    image_index = all_images_list.indexOf(ndvi_image)

    ## get the previous and next images (handling edge cases for the first/last images)
    previous_image = ee.Image(ee.Algorithms.If(image_index.gt(0), all_images_list.get(image_index.subtract(1)), None))
    next_image = ee.Image(ee.Algorithms.If(image_index.lt(all_images.size().subtract(1)), all_images_list.get(image_index.add(1)), None))

    # function to check if the NDVI values are equal to the baseline
    def is_baseline(image):
        return image.select('NDVI').eq(baseline_image.select('NDVI_baseline')).reduceRegion(
            reducer=ee.Reducer.allNonZero(),
            geometry=image.geometry(),
            scale=500
        ).values().get(0)
    
    ## check if both the previous and next images are set to the baseline
    prev_set_to_baseline = ee.Algorithms.If(previous_image, is_baseline(previous_image), ee.Number(0))
    next_set_to_baseline = ee.Algorithms.If(next_image, is_baseline(next_image), ee.Number(0))

    both_neighbors_set_to_baseline = ee.Number(prev_set_to_baseline).And(ee.Number(next_set_to_baseline))

    ## function to apply baseline if boht neighbors are set to the baseline
    def correct_to_baseline():
        return baseline_image.select('NDVI_baseline').rename('NDVI')  # Rename to 'NDVI'

    ## otherwise keep the current NDVI value
    def keep_current_ndvi():
        return ndvi_image.select('NDVI')

    ## then run it
    corrected_ndvi = ee.Image(ee.Algorithms.If(
        both_neighbors_set_to_baseline,
        correct_to_baseline(),
        keep_current_ndvi()
    ))

    ## return the NDVI image with the corrected NDVI values
    return ndvi_image.addBands(corrected_ndvi.rename('NDVI'), overwrite=True)

def apply_neighbor_check_to_collection(image_collection, baseline_image):
    """
    Apply the neighbor check to the entire collection after the initial snow correction.
    """
    ## create a list of all images in the collection to use for checking neighbors
    all_images = image_collection.sort('system:time_start')

    def check_neighbors(image):
        # Call the function to check if both neighbors are set to baseline
        corrected_image = apply_neighbor_baseline_correction(image, baseline_image, all_images)
        
        return corrected_image.select(['NDVI']).rename('NDVI')
    
    ## map the function over the entire collection
    corrected_collection = image_collection.map(check_neighbors)

    return corrected_collection


## run the neighbor baseline function
modis_ndvi_corrected = apply_neighbor_check_to_collection(modis_ndvi_corrected, winter_baseline_ndvi)



Step 3. Apply a smoothing function (moving median window with a search window of 3)

In [102]:
def apply_image_median(image_collection, window_size=3):
    
    ## define a function to apply median filtering using a fixed number of images around each image
    def smooth_image(current_image):
        
        ## get the list of all images in the collection, sorted by date
        all_images = image_collection.sort('system:time_start')
        all_images_list = all_images.toList(all_images.size())  # Convert to a list

        ## get the index of the current image
        image_index = all_images_list.indexOf(current_image)
        
        ## calculate half of the window size
        half_window = ee.Number(window_size // 2)

        ## define the start and end indices for the window (centered on the current image)
        ## ensure start_index is not less than 0
        start_index = ee.Number(image_index).subtract(half_window).max(0)

        ## ensure end_index is not greater than the total number of images
        end_index = ee.Number(image_index).add(half_window).min(all_images.size().subtract(1))

        ## create a list of images within the window
        window_images = all_images_list.slice(start_index, end_index.add(1))  # Include the end image
        window_collection = ee.ImageCollection.fromImages(window_images)

        ## compute the median of the selected images
        median_image = window_collection.median()

        ## return the median-smoothed image, keeping the original properties
        return median_image.copyProperties(current_image, current_image.propertyNames())

    ## apply the smoothing function over the entire image collection
    smoothed_collection = image_collection.map(smooth_image)

    return smoothed_collection

## apply the smoothing function to the processed NDVI collection
smoothed_ndvi = apply_image_median(modis_ndvi_corrected, window_size=3)


Export the data (at each time step so we can plot and look at them)

In [103]:
def export_collection_as_stack(image_collection, file_name, folder_name, scale=500):
    try:
        ## cast to float32
        image_collection = image_collection.map(lambda img: img.toFloat())

        ## stack all images as bands in a single multi-band image
        stacked_image = image_collection.toBands()

        ## export the stacked image to Google Drive
        task = ee.batch.Export.image.toDrive(
            image=stacked_image,
            description=f'{file_name}_stack',
            folder=folder_name,  # specify the folder in your Google Drive 
            fileNamePrefix=file_name,  # name for the exported file
            scale=scale,  # resolution in meters
            region=small_square_geom,  # region to export
            maxPixels=1e13,  # set to a high value to avoid pixel limit issues
            fileFormat='GeoTIFF'  # export as GeoTIFF
        )
        task.start()
        print(f'Exporting {file_name}_stack to Google Drive.')

    except Exception as e:
        print(f'Error exporting {file_name}: {e}')

## export the processed collections
# export_collection_as_stack(modis_ndvi.select('NDVI'), 'modis_ndvi', 'DroughtSurfing') ## only select the NDVI (so don't export the QA bands)
# export_collection_as_stack(modis_ndvi_filtered, 'modis_ndvi_filtered', 'DroughtSurfing')
export_collection_as_stack(modis_ndvi_corrected.select('NDVI'), 'modis_ndvi_corrected', 'DroughtSurfing') ## select the NDVI band
export_collection_as_stack(smoothed_ndvi, 'smoothed_ndvi', 'DroughtSurfing')

## takes about ~30 minutes to export the images for the Sublette herd area.
## Open up GEE (https://code.earthengine.google.com/) to see if it's working and it will give you errors if it doesn't work.


Exporting modis_ndvi_corrected_stack to Google Drive.
Exporting smoothed_ndvi_stack to Google Drive.
